# Spoke bronze layer : batch ingestion of "stations" csv file

## Defining variables

In [0]:
import uuid
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())

SOURCE = "/Volumes/spoke/bronze/landing/stations"
TARGET = "spoke.bronze.station"

## Reading csv file

In [0]:
df = (spark.read
      .format("csv")
      .option("delimiter", ",")
      .option("encoding", "UTF-8")
      .option("header", "true")
      .option("inferSchema", "false")
      .option("rescuedDataColumn", "_rescued_data")
      .load(SOURCE)
)

## Adding metadata columns

In [0]:
df = (df
      .withColumn("_ingested_at", F.current_timestamp())
      .withColumn("_source_file", F.col("_metadata.file_path"))
      .withColumn("_ingest_batch_id", F.lit(RUN_ID))
      .withColumn("_source_file_modification_time", F.col("_metadata.file_modification_time"))
)

## Writing to delta table

In [0]:
(df.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(TARGET)
)

## Quality checks

### Expected lines in CSV file (from raw file)

In [0]:
df_raw = (spark.read
          .format("text")
          .load(SOURCE)
)

df = (spark.read.table(TARGET))

if (df_raw.count() - 1) != df.count():
  raise Exception("Raw and decoded files have different number of rows.")

### Expected number of separators in columns (from raw file)

In [0]:
expected = len([c for c in df.columns if not c.startswith("_")])
condition = F.size(F.split(F.col("value"), ",")) != expected

df_check = df_raw.filter(condition)

if df_check.count() > 0:
  raise Exception("Unexpected number of separators in the file.")